In [0]:
# Environment variables
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate() 

In [0]:
catalog_name = "10alytics_netflex_workspace"
schema_name = "default"
volume_name = "dataset"
file_name = "Netflix_Movies_and_TV_Shows.csv"

file_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{file_name}"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(file_path)
)

display(df)

In [0]:
df = df.toDF(*[
    c.replace(' ', '_') for c in df.columns
])

In [0]:
display(df)

In [0]:
# Create table view to use sql to transform data
df.createOrReplaceTempView('movies')

In [0]:
%sql 
-- Use sql to transform data
select * from movies

In [0]:
%sql
select title, type, genre, release_year, rating, duration,
    country, concat(left(card_number, 4), '****_****') as card_number_masked
from movies

In [0]:
# Write to Delta Lake
output_path = '/mnt/data/processed/movies.csv'

transformed_df = spark.sql
(
    "select title, type, genre, release_year, rating, duration,\
    country, concat(left(card_number, 4), '****_****') as card_number_masked\
    from movies"
)

In [0]:
catalog_name = "10alytics_netflex_workspace"
schema_name = "default"
volume_name = "dataset"
file_name = "processed/Netflix_Movies_and_TV_Shows.csv"

output_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{file_name}"

In [0]:

transformed_df = spark.sql
(
    "select title, type, genre, release_year, rating, duration,\
    country, concat(left(card_number, 4), '****_****') as card_number_masked\
    from movies"
)

In [0]:
transformed_df.write.mode("overwrite").csv(
    output_path,
    header=True
)